# Old vs new tongue kinematics (library v0.5.3)

The repo moved from the pre-0.5.0 tongue pipeline (`tongue_analysis`, per-session
`intermediate_data/`, `all_tongue_movements_04022026`) to the library's kinematics core
(`docs/kinematics_core_handoff.md`). The code changes were checked locally: each notebook gives
the same numbers on the old pooled table renamed to the new columns. This notebook checks
the **data**: what the new batch output changes, session by session, before the analysis
notebooks are rerun on it.

| § | Question | Handoff |
|---|---|---|
| 1 | How many movements span a go cue, and do they all lack a trial? | check A, §2.1 |
| 2 | Which sessions pass the quality filter, old numbers vs new? Which ephys sessions are lost? | check B, §2.4 |
| 3 | How many rows do the `out_duration` filters keep? | check C, §2.5 |
| 4 | Pooled movements: same movements? How far do the features move, and how much of that is the jaw? Do `x_max` / `extent_y` reproduce `max_x_distance` / `max_y_distance`? | §1, §2.5, §2.6 |
| 5 | Ephys sessions: per-trial reaction times, cue-response movement numbers and `firstMove`, old vs new | §2.2 |
| 6 | The cached `all_counts_df`, old vs rebuilt (when both exist) | §2.2, §2.7 |

Runs on Code Ocean after the batch (`run_batch_analysis.py`, library v0.5.3) and
`build_all_tongue_movements.py`. Each section prints what it compared and skips when an input is
missing.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from aind_dynamic_foraging_behavior_video_analysis.ephys.tongue_ephys import build_event_df, get_events_dict
from data_loading import DEFAULT_COVERAGE_MIN, DEFAULT_DURATION50_MIN, passes_quality, session_quality
from ephys_utils import build_trial_features
from kin_utils import KINEMATICS_ROOT, POOLED_MOVEMENTS, POOLED_MOVEMENTS_OLD, pooled_path

SCRATCH = Path("/root/capsule/scratch")
NEW_ROOT = KINEMATICS_ROOT                         # library batch, one folder per session
# Pre-0.5.0 per-session folders: the scratch copy the eph_* notebooks used, else data asset 3f03b848
# (all 53 sessions, same movements; its tongue_movs.parquet predate the out_* backfill).
OLD_ROOT = next((p for p in [SCRATCH / "session_analysis_mlk",
                             Path("/root/capsule/data/keypoint_tracking_bottomview_LCrecordings_20260403")]
                 if p.exists()), SCRATCH / "session_analysis_mlk")
OLD_POOLED = pooled_path(POOLED_MOVEMENTS_OLD)     # all_tongue_movements_04022026
NEW_POOLED = pooled_path(POOLED_MOVEMENTS)
UNITS = SCRATCH / "combined_unit_tbl.pkl"          # ephys units; their sessions are the eph_* sessions
OLD_COUNTS = SCRATCH / "for_local" / "all_counts_df.parquet"   # cached pre-0.5.0 all_counts_df, if kept
NEW_COUNTS = SCRATCH / "all_counts_df_v053.parquet"            # written by eph_01 if saved there

def prefix(session):
    # 'behavior_751004_2024-12-20_13-26-07' -> 'behavior_751004_2024-12-20'
    return "_".join(str(session).split("_")[:3])

for name, p in [("NEW_ROOT", NEW_ROOT), ("OLD_ROOT", OLD_ROOT), ("OLD_POOLED", OLD_POOLED),
                ("NEW_POOLED", NEW_POOLED), ("UNITS", UNITS), ("OLD_COUNTS", OLD_COUNTS),
                ("NEW_COUNTS", NEW_COUNTS)]:
    print(f"{name:<11} {'ok     ' if p.exists() else 'MISSING'} {p}")
NEW_SESSIONS = sorted(p.parent.name for p in NEW_ROOT.glob("behavior_*/record.json"))
print(f"\n{len(NEW_SESSIONS)} finished sessions in NEW_ROOT")

## 1. Movements that span a go cue (check A)

Library 0.5.2 gives them no trial (`trial` NA, `spans_go_cue = True`), as before 0.5.0. Expect
about 0.01 per trial in typical sessions and up to 0.24 in 761038 (04-16, 04-18).

In [ ]:
rows = []
for s in NEW_SESSIONS:
    mv = pd.read_parquet(NEW_ROOT / s / "movements.parquet", columns=["trial", "spans_go_cue"])
    n_trials = len(pd.read_parquet(NEW_ROOT / s / "trials.parquet", columns=["trial"]))
    rows.append({"session": s, "n_spanning": int(mv["spans_go_cue"].sum()),
                 "per_trial": mv["spans_go_cue"].sum() / max(n_trials, 1),
                 "spanning_with_trial": int((mv["spans_go_cue"] & mv["trial"].notna()).sum())})
spanning = pd.DataFrame(rows).set_index("session")
print(spanning["per_trial"].describe().round(3).to_string())
print("\nhighest:\n", spanning.sort_values("per_trial", ascending=False).head(5).round(3).to_string())
print("\nspanning movements that still have a trial:", int(spanning["spanning_with_trial"].sum()), "(expect 0)")

## 2. Quality filter, old numbers vs new (check B), and the ephys sessions

Old: `tongue_quality_stats.json` in `OLD_ROOT` (`coverage_pct` 0–100 over licks that had a trial,
nearest-rank median duration). New: `record.json → quality` (`lick_coverage` 0–1 over all licks,
`median()`). Both use coverage > 0.90 and median duration > 0.06 s. A session that changes
side is listed; accept the new list deliberately.

In [ ]:
def old_quality(session_dir):
    p = Path(session_dir) / "tongue_quality_stats.json"
    if not p.exists():
        return {}
    d = json.loads(p.read_text())
    return {"cov_old": float(d.get("coverage_pct", np.nan)) / 100,
            "dur_old": float((d.get("percentiles", {}).get("duration", {}) or {}).get("0.5", np.nan))}

old_sessions = sorted(p.name for p in OLD_ROOT.glob("behavior_*")) if OLD_ROOT.exists() else []
rows = []
for s in sorted(set(old_sessions) | set(NEW_SESSIONS)):
    q = session_quality(NEW_ROOT / s) or {}
    rows.append({"session": s, "cov_new": q.get("lick_coverage", np.nan),
                 "dur_new": q.get("movement_duration_p50", np.nan), **old_quality(OLD_ROOT / s)})
quality = pd.DataFrame(rows).set_index("session")
quality["pass_new"] = [passes_quality(session_quality(NEW_ROOT / s)) for s in quality.index]
quality["pass_old"] = (quality["cov_old"] > DEFAULT_COVERAGE_MIN) & (quality["dur_old"] > DEFAULT_DURATION50_MIN)
quality["in_new_batch"] = quality.index.isin(NEW_SESSIONS)
print(pd.crosstab(quality["pass_old"], quality["pass_new"], margins=True))
changed = quality[quality["pass_old"] != quality["pass_new"]]
print(f"\n{len(changed)} sessions change side:")
print(changed.round(3).to_string())

if UNITS.exists():
    from data_loading import filter_by_criteria
    # Units passing the unit QC (data_loading.DEFAULT_CRITERIA), as in the eph_* notebooks; their
    # sessions are the ones the tongue filter can remove.
    units = filter_by_criteria(pd.read_pickle(UNITS))
    eph = sorted(set(units["session"].map(prefix)))
    by_prefix = (quality.assign(prefix=quality.index.map(prefix))
                 .groupby("prefix")[["pass_old", "pass_new", "in_new_batch"]].any())
    eph_q = by_prefix.reindex(eph).fillna(False).astype(bool)
    print(f"\nSessions with QC-passing units: {len(eph)}; pass the tongue filter: old "
          f"{int(eph_q['pass_old'].sum())}, new {int(eph_q['pass_new'].sum())}")
    lost = eph_q[eph_q["pass_old"] & ~eph_q["pass_new"]]
    print("Of those, passed before and do not now (their units leave the eph_* analyses):")
    print(lost.to_string() if len(lost) else "  none")
    print("QC-passing units in those sessions:", int(units["session"].map(prefix).isin(lost.index).sum()))

## 3. Rows kept by the `out_duration` filters (check C)

`kin_00`/`kin_01` keep `out_duration > 0.01` or `> 0.05`, and `< 5`. With the jaw at the median of
confident points, the farthest frame can move by one, so some zero-length outbound phases are
no longer zero.

In [ ]:
if OLD_POOLED.exists() and NEW_POOLED.exists():
    old_mv = pd.read_parquet(OLD_POOLED)
    new_mv = pd.read_parquet(NEW_POOLED)
    def kept(df):
        o = df["out_duration"]
        return pd.DataFrame({">0.01": o > 0.01, ">0.05": o > 0.05, "<5": o < 5, "==0": o == 0,
                             "session": df["session"]})
    k_old, k_new = kept(old_mv), kept(new_mv)
    print("Fraction of movements, pooled:")
    for col in (">0.01", ">0.05", "<5", "==0"):
        print(f"  out_duration {col:<6} old {k_old[col].mean():.3f}   new {k_new[col].mean():.3f}")
    kept_cmp = (k_old.groupby("session").mean().join(k_new.groupby("session").mean(),
                lsuffix="_old", rsuffix="_new", how="outer"))
    print("\nPer session (sessions in both):")
    print(kept_cmp.dropna().round(3).to_string())
else:
    print("[skip] needs both pooled tables")

## 4. Pooled movements, matched one to one

Movements are matched by session, start and end time (rounded to 0.1 ms; same clock before and
after). The old positions are absolute pixels; they are put in the old jaw's frame (the old jaw,
the mean of all jaw points, is recovered from `max_x_from_jaw - max_x_distance`). The new ones are
relative to the new jaw (median of points with likelihood ≥ 0.9). Shifting the new values by the
jaw difference puts both in the old frame, so what remains is not the jaw. Expect: identical
movements; positions within ~1 px after the jaw shift; `peak_speed` a median ~8% different
(smoothing per tracked run, §2.6); `|x_max|` = `max_x_distance` and `|extent_y|` =
`max_y_distance` up to the jaw shift.

In [ ]:
def old_jaw(old):
    # old jaw per session: max_x_distance = |x - jaw| at the most positive x offset
    cx = (old["max_x_from_jaw"] - old["max_x_distance"]).round(4)
    cy = pd.concat([(old["max_y_from_jaw"] - old["max_y_distance"]).round(4),
                    (old["max_y_from_jaw"] + old["max_y_distance"]).round(4)])
    sess = pd.concat([old["session"], old["session"]])
    return pd.DataFrame({"jaw_x_old": cx.groupby(old["session"]).agg(lambda s: s.mode().iloc[0]),
                         "jaw_y_old": cy.groupby(sess).agg(lambda s: s.mode().iloc[0])})

if OLD_POOLED.exists() and NEW_POOLED.exists():
    from kin_utils import POOLED_LANDMARKS
    lm = pd.read_parquet(pooled_path(POOLED_LANDMARKS))
    jaw = old_jaw(old_mv).join(lm[lm["keypoint"] == "jaw"].set_index("session")[["x", "y"]]
                               .rename(columns={"x": "jaw_x_new", "y": "jaw_y_new"}), how="inner")
    jaw["dx"], jaw["dy"] = jaw["jaw_x_new"] - jaw["jaw_x_old"], jaw["jaw_y_new"] - jaw["jaw_y_old"]
    print("Jaw shift new - old (px):\n", jaw[["dx", "dy"]].describe().round(2).to_string())

    key = lambda d: d.assign(_s=d["start_time"].round(4), _e=d["end_time"].round(4))
    m = key(old_mv).merge(key(new_mv), on=["session", "_s", "_e"], suffixes=("_o", "_n"))
    both = sorted(set(old_mv["session"]) & set(new_mv["session"]))
    n_old = old_mv["session"].isin(both).sum(); n_new = new_mv["session"].isin(both).sum()
    print(f"\nSessions in both: {len(both)}. Movements: old {n_old:,}, new {n_new:,}, matched {len(m):,}")

    m = m.join(jaw[["jaw_x_old", "jaw_y_old", "dx", "dy"]], on="session")
    # (old column in absolute px or as-is, new column, how to put the new one in the old frame)
    pairs = [
        ("duration_o", "duration_n", None), ("n_datapoints", "n_valid", None),
        ("total_distance", "path_length", None), ("peak_velocity", "peak_speed", None),
        ("mean_velocity", "mean_speed", None), ("out_duration_o", "out_duration_n", None),
        ("out_peak_velocity", "out_peak_speed", None), ("out_total_distance", "out_path_length", None),
        ("endpoint_x", "end_x", "x"), ("endpoint_y", "end_y", "y"),
        ("startpoint_x", "start_x", "x"), ("startpoint_y", "start_y", "y"),
        ("max_x_from_jaw", "x_max", "x"), ("max_x_from_jaw_y", "y_at_x_max", "y"),
        ("max_y_from_jaw", "extent_y", "y"), ("max_y_from_jaw_x", "x_at_extent_y", "x"),
    ]
    rows = []
    for o, n, axis in pairs:
        if o not in m or n not in m:
            continue
        old_v = m[o].astype(float)
        new_v = m[n].astype(float)
        if axis == "x":
            old_v = old_v - m["jaw_x_old"]; new_v = new_v + m["dx"]
        elif axis == "y":
            old_v = old_v - m["jaw_y_old"]; new_v = new_v + m["dy"]
        d = (new_v - old_v).abs()
        rel = d / old_v.abs().where(old_v.abs() > 0)
        rows.append({"old": o, "new": n, "n": int(d.notna().sum()), "median_abs_diff": d.median(),
                     "p99_abs_diff": d.quantile(0.99), "median_rel_diff": rel.median(),
                     "p99_rel_diff": rel.quantile(0.99), "exact_share": float((d.dropna() < 1e-6).mean())})
    print("\nMatched movements, new vs old (positions in the old jaw frame):")
    print(pd.DataFrame(rows).round(4).to_string(index=False))

    # The distance columns as the library's signed extremes, each against its own jaw.
    dist = pd.DataFrame({
        "|x_max| vs max_x_distance": (m["x_max"].abs() - m["max_x_distance"]).abs(),
        "|extent_y| vs max_y_distance": (m["extent_y"].abs() - m["max_y_distance"]).abs(),
        "|extent_x| vs max_x_distance": (m["extent_x"].abs() - m["max_x_distance"]).abs(),
    })
    print("\nDistance columns, each relative to its own jaw (|diff| px):")
    print(dist.describe(percentiles=[0.5, 0.9, 0.99]).round(3).to_string())
    print("share of |extent_x| != max_x_distance by > 1 px (handoff: ~12%):",
          round(float((dist["|extent_x| vs max_x_distance"] > 1).mean()), 3))

## 5. Ephys sessions: per-trial features and `firstMove`

For every ephys session in both roots: the pre-0.5.0 `build_trial_features` (copied below from the
repo before the migration) on the old tables, against the current one on the new tables; and
`firstMove` from the old per-frame rule against the new per-movement one. Expected differences:
trials with a movement that spans the go cue (`firstMove` moves from ~2 ms to the next movement's
start, §2.2), the 2025 sessions whose trials were mostly missing before (§2.3), and the last
trial (§2.10).

In [ ]:
def old_build_trial_features(movs):
    """build_trial_features before the 0.5.x migration (repo commit c54bdc7~1), on tongue_movs.parquet."""
    tm = movs.copy()
    crmn = tm.groupby("trial")["cue_response_movement_number"].first().astype("Int64")
    rt_first = tm.groupby("trial")["movement_latency_from_go"].min()
    rt_cr = tm.loc[tm["cue_response"] == True].groupby("trial")["movement_latency_from_go"].first()
    out = pd.DataFrame({"cue_response_movement_number": crmn,
                        "reaction_time_firstmove": rt_first,
                        "reaction_time_cueresponse": rt_cr})
    kcols = [c for c in ["peak_velocity", "duration", "excursion_angle_deg", "endpoint_x", "endpoint_y",
                         "out_duration", "out_peak_velocity", "out_mean_velocity", "out_total_distance"]
             if c in tm]   # the data asset's copies have no out_*
    first = (tm.loc[tm["movement_number_in_trial"] == 1, ["trial", *kcols]]
             .drop_duplicates("trial").set_index("trial").add_prefix("first_move_"))
    return out.join(first, how="left").sort_index()


def old_first_move(session_dir):
    """firstMove before 0.5.0: the first frame after the go cue of the trial's first movement,
    from the per-frame table (frames carry their trial)."""
    inter = Path(session_dir) / "intermediate_data"
    trials = pd.read_parquet(inter / "nwb_df_trials.parquet")
    kins = pd.read_parquet(inter / "tongue_kins.parquet", columns=["time_in_session", "trial", "movement_id"])
    E = build_event_df({"goCue": (trials, "goCue_start_time_in_session", "trial"),
                        "firstMove": (kins.dropna(subset=["movement_id"]), "time_in_session", "trial",
                                      {"min_after_cue": True})})
    return E


def new_first_move(session_dir):
    s = Path(session_dir)
    trials, licks, movs = (pd.read_parquet(s / f"{n}.parquet") for n in ("trials", "licks", "movements"))
    return build_event_df(get_events_dict(trials, licks, movs))[["goCue", "firstMove"]]


if UNITS.exists() and OLD_ROOT.exists():
    eph_prefixes = set(units["session"].map(prefix))
    sessions = [s for s in NEW_SESSIONS if prefix(s) in eph_prefixes and (OLD_ROOT / s / "intermediate_data").exists()]
    rows, fm_rows = [], []
    for s in sessions:
        inter = OLD_ROOT / s / "intermediate_data"
        old_tf = old_build_trial_features(pd.read_parquet(inter / "tongue_movs.parquet"))
        new_s = NEW_ROOT / s
        new_tf = build_trial_features(pd.read_parquet(new_s / "movements.parquet"),
                                      pd.read_parquet(new_s / "licks.parquet"),
                                      pd.read_parquet(new_s / "trials.parquet"))
        j = old_tf.join(new_tf, how="outer", lsuffix="_old", rsuffix="_new")
        both = j.dropna(subset=["reaction_time_firstmove_old", "reaction_time_firstmove_new"])
        rows.append({
            "session": s, "trials_old": len(old_tf), "trials_new": len(new_tf), "trials_both": len(both),
            "rt_first_same": float((both["reaction_time_firstmove_old"] - both["reaction_time_firstmove_new"]).abs().lt(1e-3).mean()),
            "rt_cue_same": float((j["reaction_time_cueresponse_old"] - j["reaction_time_cueresponse_new"]).abs().lt(1e-3)
                                 [j[["reaction_time_cueresponse_old", "reaction_time_cueresponse_new"]].notna().all(axis=1)].mean()),
            "crmn_same": float((j["cue_response_movement_number_old"] == j["cue_response_movement_number_new"])
                               [j[["cue_response_movement_number_old", "cue_response_movement_number_new"]].notna().all(axis=1)].mean()),
            "crmn_trials_old": int(j["cue_response_movement_number_old"].notna().sum()),
            "crmn_trials_new": int(j["cue_response_movement_number_new"].notna().sum()),
        })
        try:
            fo, fn = old_first_move(OLD_ROOT / s), new_first_move(new_s)
            f = fo.join(fn, lsuffix="_old", rsuffix="_new", how="inner")
            lat_o, lat_n = f["firstMove_old"] - f["goCue_old"], f["firstMove_new"] - f["goCue_new"]
            fm_rows.append({"session": s, "trials": len(f),
                            "same": float((lat_o - lat_n).abs().lt(1e-3).mean()),
                            "old_within_5ms": float(lat_o.lt(0.005).mean()),
                            "new_within_5ms": float(lat_n.lt(0.005).mean()),
                            "missing_old": int(lat_o.isna().sum()), "missing_new": int(lat_n.isna().sum())})
        except Exception as e:  # noqa: BLE001
            print(f"[firstMove skip] {s}: {e!r}")
    trial_cmp = pd.DataFrame(rows).set_index("session")
    print("Per-trial features, old vs new (share of trials in both with the same value, |diff| < 1 ms):")
    print(trial_cmp.round(3).to_string())
    if fm_rows:
        print("\nfirstMove latency, old vs new:")
        print(pd.DataFrame(fm_rows).set_index("session").round(3).to_string())
else:
    print("[skip] needs combined_unit_tbl.pkl and OLD_ROOT")

## 6. `all_counts_df`, old vs rebuilt

When `eph_01` has rebuilt `all_counts_df` and saved it to `NEW_COUNTS`, and the pre-0.5.0 cache
is at `OLD_COUNTS`: unit and trial coverage, and agreement of spike counts (which should not
move) and reaction times (which should only move where §5 says).

In [ ]:
if OLD_COUNTS.exists() and NEW_COUNTS.exists():
    oc, nc = pd.read_parquet(OLD_COUNTS), pd.read_parquet(NEW_COUNTS)
    k = ["session", "unit_id", "trial"]
    j = oc.merge(nc, on=k, how="outer", suffixes=("_old", "_new"), indicator=True)
    print(j["_merge"].value_counts().to_string())
    b = j[j["_merge"] == "both"]
    for c in ["spike_count", "baseline_spike_count", "reaction_time_firstmove", "reaction_time_cueresponse"]:
        if f"{c}_old" in b and f"{c}_new" in b:
            same = (b[f"{c}_old"] - b[f"{c}_new"]).abs().lt(1e-3) | (b[f"{c}_old"].isna() & b[f"{c}_new"].isna())
            print(f"{c:<28} same in {same.mean():.4f} of {len(b):,} unit-trials")
    print("\nUnits only in the old table, by session:")
    print(j[j["_merge"] == "left_only"].groupby("session")["unit_id"].nunique().to_string())
else:
    print("[skip] needs both all_counts tables (save eph_01's all_counts_df to NEW_COUNTS)")